## tl;dr

Pundits.Pro has a credible content-supply wedge: every verified mapped pick creates one canonical receipt page and two social image formats, then grading refreshes the same assets with an outcome. In the base four-week scenario (100 new mapped picks and 35 newly covered events per week), the site adds 577 indexable pages, grows from 117 to 694 indexable URLs, and produces 2,160 lifecycle media renditions. These are projections, not observed traffic forecasts.

## Context & Methods

Decision: whether to prioritize content volume before monetization and growth. The analysis counts current repository records and generated routes, then projects three four-week pick-supply scenarios.

### Key Assumptions

- A new mapped pick creates one canonical take page.
- A newly covered event creates one event page.
- Two sport/week archive pages are added per week.
- Pundit and team profile activations are capped by the current unindexed roster: 28 pundits and 9 teams.
- Each pick and event has two media formats (OG/link preview and vertical story). Grading regenerates both formats, so lifecycle renditions are four per pick/event but only two current file paths.
- Feed entries, schema, sitemap changes, and page refreshes are distribution/SEO signals, not additional canonical pages.

## Data

Sources: `data/calls.json`, `data/events.json`, `data/pundits.json`, `data/teams.json`, generated `out/sitemap.xml`, and the route/image logic in `app/sitemap.ts`, `lib/seo.ts`, and `scripts/render-og.tsx`. Repository snapshot reviewed August 28, 2026.

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET

repo = Path.cwd()
calls = json.loads((repo / 'data/calls.json').read_text(encoding='utf-8'))
events = json.loads((repo / 'data/events.json').read_text(encoding='utf-8'))['events']
pundits = json.loads((repo / 'data/pundits.json').read_text(encoding='utf-8'))
teams = json.loads((repo / 'data/teams.json').read_text(encoding='utf-8'))
mapped = [c for c in calls if c.get('eventSlug') and c.get('side')]

root = ET.parse(repo / 'out/sitemap.xml').getroot()
ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
urls = [node.text for node in root.findall('sm:url/sm:loc', ns)]

baseline = {
    'captured_calls': len(calls),
    'mapped_picks': len(mapped),
    'settled_mapped_picks': sum(c.get('status') in {'hit', 'miss'} for c in mapped),
    'events': len(events),
    'rostered_pundits': len(pundits),
    'indexable_urls': len(urls),
    'generated_social_images': sum(1 for p in (repo / 'public/og').rglob('*.png')),
}
baseline

{'captured_calls': 57,
 'mapped_picks': 34,
 'settled_mapped_picks': 0,
 'events': 31,
 'rostered_pundits': 48,
 'indexable_urls': 117,
 'generated_social_images': 226}

## Results

In [2]:
weeks = 4
current_urls = baseline['indexable_urls']
scenarios = [
    {'scenario': 'Focused', 'picks_per_week': 50, 'events_per_week': 20, 'new_pundit_pages': 10, 'new_team_pages': 5},
    {'scenario': 'Base', 'picks_per_week': 100, 'events_per_week': 35, 'new_pundit_pages': 20, 'new_team_pages': 9},
    {'scenario': 'Breakout', 'picks_per_week': 200, 'events_per_week': 55, 'new_pundit_pages': 28, 'new_team_pages': 9},
]

projection = []
for row in scenarios:
    picks = row['picks_per_week'] * weeks
    new_events = row['events_per_week'] * weeks
    archives = 2 * weeks
    new_pages = picks + new_events + archives + row['new_pundit_pages'] + row['new_team_pages']
    current_media_files = 2 * (picks + new_events)
    lifecycle_media_renditions = 4 * (picks + new_events)
    potential_post_units = 2 * picks + 2 * new_events + (3 * weeks)
    projection.append({
        **row,
        'four_week_picks': picks,
        'new_event_pages': new_events,
        'new_indexable_pages': new_pages,
        'projected_indexable_urls': current_urls + new_pages,
        'new_current_media_files': current_media_files,
        'lifecycle_media_renditions': lifecycle_media_renditions,
        'potential_post_units': potential_post_units,
    })

columns = ['scenario', 'four_week_picks', 'new_event_pages', 'new_indexable_pages', 'projected_indexable_urls', 'new_current_media_files', 'lifecycle_media_renditions', 'potential_post_units']
print(' | '.join(columns))
print(' | '.join(['---'] * len(columns)))
for row in projection:
    print(' | '.join(str(row[c]) for c in columns))

scenario | four_week_picks | new_event_pages | new_indexable_pages | projected_indexable_urls | new_current_media_files | lifecycle_media_renditions | potential_post_units
--- | --- | --- | --- | --- | --- | --- | ---
Focused | 200 | 80 | 303 | 420 | 560 | 1120 | 572
Base | 400 | 140 | 577 | 694 | 1080 | 2160 | 1092
Breakout | 800 | 220 | 1065 | 1182 | 2040 | 4080 | 2052


In [3]:
# Independent arithmetic checks for the base scenario.
base = next(r for r in projection if r['scenario'] == 'Base')
assert base['new_indexable_pages'] == 400 + 140 + 8 + 20 + 9 == 577
assert base['projected_indexable_urls'] == 117 + 577 == 694
assert base['new_current_media_files'] == 2 * (400 + 140) == 1080
assert base['lifecycle_media_renditions'] == 4 * (400 + 140) == 2160
assert base['potential_post_units'] == 2 * 400 + 2 * 140 + 12 == 1092
print('Base-scenario arithmetic verified.')

Base-scenario arithmetic verified.


## Takeaways

- The base case creates about 577 new canonical pages in four weeks, not thousands. The thousands figure belongs to page refreshes and media renditions.
- Volume should be judged by settled sample depth: hundreds of graded picks, at least 10 settled picks for a meaningful set of pundits, and dense coverage of high-interest events.
- Publishing every possible social unit would be operationally noisy. Automate the complete receipt/result stream on X, but curate vertical/social feeds around surprises, disagreements, streaks, and weekly scoreboards.
- The strongest moat is verified source evidence plus immutable grading history. Raw page count is easy for competitors to copy.
- Monetization should remain light until repeat use and search indexing are visible, while email capture and sponsor-interest tests can begin immediately.